In [1]:
import os
os.chdir(r'C:\Users\johnpaul\fraudguard-africa')
print(os.getcwd())

C:\Users\johnpaul\fraudguard-africa


In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# ====================== LOAD DATA ======================
df = pd.read_csv('data/PS_20174392719_1491204439457_log.csv')
df = df.sample(frac=0.30, random_state=42).reset_index(drop=True)   # 30% for speed
print(df.shape)

(1908786, 11)


In [3]:
# ====================== FEATURE ENGINEERING ======================
df['balance_diff_orig'] = df['oldbalanceOrg'] - df['newbalanceOrig']
df['balance_diff_dest'] = df['oldbalanceDest'] - df['newbalanceDest']
df['amount_to_oldbalance_ratio'] = df['amount'] / (df['oldbalanceOrg'] + 1e-8)
df['amount_to_newbalance_ratio'] = df['amount'] / (df['newbalanceOrig'] + 1e-8)

df['hour'] = df['step'] % 24
df['is_night'] = df['hour'].isin([0,1,2,3,4,5,22,23]).astype(int)

df = pd.get_dummies(df, columns=['type'], prefix='type', drop_first=True)

orig_freq = df['nameOrig'].value_counts()
df['orig_transaction_freq'] = df['nameOrig'].map(orig_freq)

df['high_risk_transaction'] = ((df['type_CASH_OUT'] == 1) & (df['amount'] > 100000)).astype(int)

# ====================== PREPARE X, y ======================
drop_cols = ['nameOrig', 'nameDest', 'isFlaggedFraud', 'step']
feature_cols = [col for col in df.columns if col not in drop_cols + ['isFraud']]

X = df[feature_cols]
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                    random_state=42, stratify=y)

In [4]:
# SMOTE
smote = SMOTE(random_state=42, sampling_strategy=0.1)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Training samples after SMOTE:", X_train_res.shape)

Training samples after SMOTE: (1677588, 17)


In [5]:
# ====================== MODELS ======================
models = {
    "XGBoost": xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                                 subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1),
    
    "LightGBM": lgb.LGBMClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                                   class_weight='balanced', random_state=42, n_jobs=-1),
    
    "RandomForest": RandomForestClassifier(n_estimators=80, max_depth=7, 
                                           class_weight='balanced', random_state=42, n_jobs=-1)
}

# ====================== TRAINING & EVALUATION ======================
for name, model in models.items():
    print(f"\n=== Training {name} ===")
    model.fit(X_train_res, y_train_res)
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    print(classification_report(y_test, y_pred, digits=4))
    print(f"ROC-AUC : {roc_auc_score(y_test, y_pred_proba):.4f}")
    
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    print(f"PR-AUC  : {auc(recall, precision):.4f}")
    print("-" * 70)


=== Training XGBoost ===
              precision    recall  f1-score   support

           0     1.0000    0.9993    0.9996    381271
           1     0.6498    0.9754    0.7800       487

    accuracy                         0.9993    381758
   macro avg     0.8249    0.9873    0.8898    381758
weighted avg     0.9995    0.9993    0.9994    381758

ROC-AUC : 0.9983
PR-AUC  : 0.9570
----------------------------------------------------------------------

=== Training LightGBM ===
[LightGBM] [Info] Number of positive: 152508, number of negative: 1525080
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.169519 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2334
[LightGBM] [Info] Number of data points in the train set: 1677588, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[Light